[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/semantica-agi/semantica/blob/main/cookbook/introduction/25_Seed_Data.ipynb)

# 种子数据 —— 实用指南

`seed` 模块在运行任何抽取之前,基于**可信的、预先已知的数据**(CSV/JSON/数据库/API 来源)引导构建知识图谱。这为抽取提供了可供链接的基础,而不是从一张空图开始。

关键组成部分:

- **`SeedDataManager`** —— 注册数据源并构建基础图
- **`create_foundation_graph()`** —— 将已注册的数据源转换为 `entities` + `relationships` + `metadata`
- **`validate_quality()`** —— 在提交基础图之前对其进行检查

下面所有示例都是针对 `semantica/seed/seed_manager.py` 执行的。

In [1]:
!pip install -q semantica

## 1)准备种子 CSV 并注册数据源

`register_source(name, format, location, entity_type=...)` 记录可信数据所在的位置。`verified=True`(默认值)表示该数据源已预先通过验证。

In [2]:
import csv
import tempfile
from pathlib import Path
from semantica.seed import SeedDataManager

# Write the sample CSV into a session-scoped temp directory so we never
# clobber a companies.csv that might exist in the user's working directory.
seed_csv = Path(tempfile.mkdtemp(prefix="semantica-seed-")) / "companies.csv"
with open(seed_csv, "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=["id", "name", "type", "industry"])
    writer.writeheader()
    writer.writerow({"id": "c1", "name": "Acme", "type": "Company", "industry": "robotics"})
    writer.writerow({"id": "c2", "name": "Globex", "type": "Company", "industry": "energy"})

manager = SeedDataManager()
manager.register_source("companies", format="csv", location=str(seed_csv), entity_type="Company")


True

## 2)从已注册的数据源加载记录

`load_source(name)` 读取数据源,并为每条记录补充 `entity_type` 和 `source` 溯源键。

In [3]:
records = manager.load_source("companies")
print(f"loaded {len(records)} records")
records[0]

Status,Action,Module,Submodule,Progress,ETA,Rate,Time,Extracted
✅,Semantica is seeding,🌱 seed,SeedDataManager,100.0%,-,-,0.00s,-


🔄 Semantica is seeding: Loading seed data from CSV: /var/folders/7s/bvvstgs10y963tz6_4bbnklr0000gn/T/semantica-seed-eu9__ep1/companies.csv 🌱 seed SeedDataManager |░░░░░░░░░░░░░░░| 0.0% ETA: - Rate: - Time: 0.00s Extracted: -

loaded 2 records


{'id': 'c1',
 'name': 'Acme',
 'type': 'Company',
 'industry': 'robotics',
 'entity_type': 'Company',
 'source': 'companies'}

## 3)构建基础图

`create_foundation_graph()` 会把每个已注册的数据源转换为可直接入图的实体和关系。实体带有 `confidence: 1.0` —— 种子数据按定义就是可信的。

In [4]:
foundation = manager.create_foundation_graph()
sorted(foundation.keys())

['entities', 'metadata', 'relationships']

In [5]:
foundation["entities"][0]

{'id': 'c1',
 'text': 'Acme',
 'type': 'Company',
 'confidence': 1.0,
 'metadata': {'industry': 'robotics', 'source': 'companies'}}

## 4)提交前验证质量

`validate_quality(foundation_graph)` 返回 `valid`、`errors`、`warnings` 和 `metrics`,使你可以在错误的种子数据污染图之前拦截它们。

In [6]:
quality = manager.validate_quality(foundation)
quality["valid"]

True

## 总结

| 任务 | API |
|---|---|
| 注册可信数据源 | `register_source(name, format, location, entity_type=...)` |
| 加载记录 | `load_source(name)` —— 添加 `entity_type` / `source` 键 |
| 直接加载文件 | `load_from_csv(path)` / `load_from_json(path)` |
| 构建图 | `create_foundation_graph()` → `entities` / `relationships` / `metadata` |
| 拦截错误数据 | `validate_quality(graph)` → `valid` / `errors` / `warnings` / `metrics` |

另请参阅 `semantica/seed/seed_usage.md`,了解 `load_from_database`、`load_from_api` 和 `integrate_with_extracted`。